# Priya TTS — GPU service only (IndicF5)

Runs ONLY the IndicF5 voice service on a Kaggle GPU and exposes it through a
public tunnel. Everything else (website, database, voicebot, Whisper STT,
Groq brain) runs on YOUR LAPTOP — see the hybrid setup in the repo README.

**How to use:** Accelerator = any GPU, Internet = ON, then Run All (~10 min).
When the last cell prints the tunnel URL, paste it into the laptop `.env` as
`TTS_SERVICE_URL=...` and restart the laptop's voicebot + website.

**Secrets needed** (Add-ons → Secrets): `GITHUB_TOKEN`, `WHATSAPP_SERVICE_KEY`, `HF_TOKEN`
(must be the SAME value as in the laptop .env — it authenticates TTS requests).


`HF_TOKEN`: IndicF5 is a GATED HuggingFace model. Request access at
https://huggingface.co/ai4bharat/IndicF5 (click Agree), then create a Read
token at hf.co → Settings → Access Tokens and save it as this secret.


In [ ]:
%%bash
# ---- 1. cloudflared (for the public tunnel) ----
set -e
curl -sL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
echo "cloudflared OK"


In [ ]:
# ---- 2. Clone the repo (tts-service code + Priya's reference voice) ----
from kaggle_secrets import UserSecretsClient
import subprocess

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
service_key = secrets.get_secret("WHATSAPP_SERVICE_KEY")
hf_token = secrets.get_secret("HF_TOKEN")  # IndicF5 is gated on HF

subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1",
     f"https://{github_token}@github.com/arjungaming371-cmyk/right-agent-group.git",
     "/kaggle/working/app"],
    check=True,
)

# Minimal .env — the tts-service reads TTS_API_KEY (or WHATSAPP_SERVICE_KEY)
# from /kaggle/working/app/.env to authenticate incoming requests.
with open("/kaggle/working/app/.env", "w", encoding="utf-8") as f:
    f.write("TTS_API_KEY=" + service_key + chr(10))
    f.write("HF_TOKEN=" + hf_token + chr(10))

print("repo cloned, .env written")


In [ ]:
%%bash
# ---- 3. Install IndicF5 (a few minutes) ----
set -eo pipefail
pip install -q git+https://github.com/ai4bharat/IndicF5.git soundfile 2>&1 | tail -4
# Newer transformers inits models on the meta device, which IndicF5
# custom code cannot survive — pin a known-good version LAST so nothing
# above re-upgrades it.
pip install -q "transformers==4.47.1" 2>&1 | tail -2
echo "--- install done ---"


In [ ]:
%%bash
# ---- 4. Start the TTS service on the GPU + poll until warm ----
set -e
cd /kaggle/working/app/server/tts-service
export HF_TOKEN=$(grep -E '^HF_TOKEN=' /kaggle/working/app/.env | cut -d= -f2)
# TORCHDYNAMO_DISABLE: IndicF5 wraps its model in torch.compile, which
# RECOMPILES for every new sentence length (~15s per call). Eager mode
# skips compilation entirely — slightly slower kernels, no recompile tax.
CUDA_VISIBLE_DEVICES=0 TORCHDYNAMO_DISABLE=1 TORCH_COMPILE_DISABLE=1 HUGGING_FACE_HUB_TOKEN="$HF_TOKEN" nohup python -u -m uvicorn app:app --host 127.0.0.1 --port 3004 > /kaggle/working/tts.log 2>&1 &
echo "started, pid $!"
KEY=$(grep -E '^TTS_API_KEY=' /kaggle/working/app/.env | cut -d= -f2)
# First start downloads ~1.5GB from HF + loads + warmup synthesis.
UP=0
for i in $(seq 1 60); do
  sleep 10
  if curl -sf -o /dev/null --max-time 3 -H "x-api-key: $KEY" http://127.0.0.1:3004/health 2>/dev/null; then
    echo "[${i}0s] TTS responding"
    UP=1
    break
  fi
  echo "[${i}0s] not up yet..."
done
if [ "$UP" != "1" ]; then
  echo "TTS never came up after 10 min — tts.log follows:"
  tail -60 /kaggle/working/tts.log
  exit 1
fi
echo "--- last 20 lines of tts.log ---"
tail -20 /kaggle/working/tts.log


In [ ]:
%%bash
# ---- 5. Real synthesis gate (Telugu) — fail loudly if the voice is broken ----
set -e
KEY=$(grep -E '^TTS_API_KEY=' /kaggle/working/app/.env | cut -d= -f2)
cat > /tmp/tts_test_body.json << 'JSONEOF'
{"text":"నమస్కారం! నేను ప్రియ. మీకు లోన్ గురించి సహాయం చేస్తాను."}
JSONEOF
echo "--- synthesizing (timing it) ---"
time curl -s -X POST http://127.0.0.1:3004/synthesize -H "Content-Type: application/json" -H "x-api-key: $KEY" -d @/tmp/tts_test_body.json -o /kaggle/working/tts-test.wav
SIZE=$(stat -c%s /kaggle/working/tts-test.wav 2>/dev/null || echo 0)
echo "--- result: ${SIZE} bytes ---"
if [ "$SIZE" -lt 10000 ]; then
  echo "Only ${SIZE} bytes — an error, not audio. Check tts.log above."
  head -c 300 /kaggle/working/tts-test.wav; echo ''
  exit 1
fi
echo "IndicF5 confirmed working."


In [ ]:
# ---- 6. Public tunnel — paste this URL into the laptop .env ----
import subprocess, re, time

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3004", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("tunnel URL not found — rerun this cell")

with open("/kaggle/working/tts_url.txt", "w") as f:
    f.write(url)

print("=" * 60)
print("ON YOUR LAPTOP — put this line in .env:")
print(f"  TTS_SERVICE_URL={url}")
print("then restart the website + voicebot. Keep this notebook running.")
print("=" * 60)
